In [0]:
print("\n" + "="*80)
print("RESULTADOS DE LA OPTIMIZACIÓN")
print("="*80)

# Mejor trial
best_trial = study.best_trial

print(f"\n🏆 Mejores Hiperparámetros Encontrados:")
for key, value in best_trial.params.items():
    print(f"   {key}: {value}")

print(f"\n📊 Mejor MAE (validación cruzada): ${best_trial.value:,.2f}")
print(f"\n📉 Mejora vs. Baseline: ${mae_baseline - best_trial.value:,.2f} ({((mae_baseline - best_trial.value)/mae_baseline)*100:.2f}%)")

# Estadísticas del estudio
print(f"\n📊 Estadísticas del Estudio:")
print(f"   Total de trials: {len(study.trials)}")
print(f"   Trials completos: {len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])}")
print(f"   Mejor trial: #{best_trial.number}")

In [0]:
print("\n" + "="*80)
print("MODELO FINAL CON HIPERPARÁMETROS OPTIMIZADOS")
print("="*80)

# Entrenar modelo con mejores hiperparámetros
model_optimized = RandomForestRegressor(**best_trial.params)
model_optimized.fit(X_train, y_train)

# Evaluar en test set
y_pred_optimized = model_optimized.predict(X_test)

mae_optimized = mean_absolute_error(y_test, y_pred_optimized)
rmse_optimized = np.sqrt(mean_squared_error(y_test, y_pred_optimized))
r2_optimized = r2_score(y_test, y_pred_optimized)

print(f"\n📊 Métricas en Test Set:")
print(f"   MAE:  ${mae_optimized:,.2f}")
print(f"   RMSE: ${rmse_optimized:,.2f}")
print(f"   R²:   {r2_optimized:.4f}")

# Comparación
print(f"\n🏆 Comparación Baseline vs. Optimizado:")
print(f"\n   {'Métrica':<10} {'Baseline':<15} {'Optimizado':<15} {'Mejora'}")
print(f"   {'-'*10} {'-'*15} {'-'*15} {'-'*15}")
print(f"   {'MAE':<10} ${mae_baseline:<14,.2f} ${mae_optimized:<14,.2f} {((mae_baseline-mae_optimized)/mae_baseline)*100:>6.2f}%")
print(f"   {'RMSE':<10} ${rmse_baseline:<14,.2f} ${rmse_optimized:<14,.2f} {((rmse_baseline-rmse_optimized)/rmse_baseline)*100:>6.2f}%")
print(f"   {'R²':<10} {mae_baseline:<15.4f} {r2_optimized:<15.4f} {((r2_optimized-r2_baseline)/r2_baseline)*100:>6.2f}%")

In [0]:
print("\n" + "="*80)
print("VISUALIZACIONES DE OPTIMIZACIÓN")
print("="*80)

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Gráfico 1: Historia de optimización
fig = optuna.visualization.plot_optimization_history(study)
fig.update_layout(
    title="Historia de Optimización: Mejor MAE por Trial",
    xaxis_title="Número de Trial",
    yaxis_title="MAE ($)",
    height=500
)
fig.show()

print("\n📊 Gráfico 1: Historia de Optimización")
print("   Muestra cómo mejora el mejor MAE encontrado a lo largo de los trials")
print("   La línea azul es el mejor valor acumulado")

In [0]:
# Gráfico 2: Importancia de hiperparámetros
fig = optuna.visualization.plot_param_importances(study)
fig.update_layout(
    title="Importancia de Hiperparámetros",
    xaxis_title="Importancia",
    height=500
)
fig.show()

print("\n📊 Gráfico 2: Importancia de Hiperparámetros")
print("   Muestra qué hiperparámetros tienen mayor impacto en el resultado")
print("   Los más importantes son los que más deberías ajustar con cuidado")

In [0]:
# Gráfico 3: Slice plot (relación individual)
fig = optuna.visualization.plot_slice(study)
fig.update_layout(
    title="Relación entre Hiperparámetros y MAE",
    height=600
)
fig.show()

print("\n📊 Gráfico 3: Slice Plot")
print("   Muestra cómo cada hiperparámetro afecta individualmente al MAE")
print("   Puntos azules: trials individuales")
print("   Ayuda a entender rangos óptimos para cada parámetro")

## 5️⃣ Técnicas Avanzadas con Optuna

### ✏️ Pruning Automático

**Concepto**: Detener trials poco prometedores **antes de terminar** el entrenamiento completo.

**Ejemplo**: Si después de 10 epochs tu modelo tiene peor loss que el 75% de trials anteriores, deténlo.

✅ **Ventaja**: Ahorra tiempo al no entrenar modelos malos hasta el final.

```python
import optuna
from optuna.pruners import MedianPruner

# Crear estudio con pruning
study = optuna.create_study(
    pruner=MedianPruner(
        n_startup_trials=5,  # No pruning en primeros 5 trials
        n_warmup_steps=3     # Esperar 3 steps antes de pruning
    )
)
```

**Tipos de Pruners**:
- **MedianPruner**: Poda si peor que mediana de trials
- **PercentilePruner**: Poda si peor que percentil X
- **SuccessiveHalvingPruner**: Estilo torneo (elimina mitad peor iterativamente)

---

### 🚀 Paralelización

**Concepto**: Ejecutar múltiples trials **simultáneamente** en varios cores.

```python
# Opción 1: Con joblib (varios procesos)
study.optimize(objective, n_trials=100, n_jobs=4)  # 4 procesos paralelos

# Opción 2: Múltiples scripts apuntando a misma DB
# Script 1:
study = optuna.load_study(
    study_name='mi_estudio',
    storage='sqlite:///optuna.db'
)
study.optimize(objective, n_trials=50)

# Script 2 (en paralelo):
study = optuna.load_study(
    study_name='mi_estudio',
    storage='sqlite:///optuna.db'
)
study.optimize(objective, n_trials=50)
```

---

### 💾 Persistencia en Base de Datos

**Concepto**: Guardar progreso para reanudar o analizar después.

```python
# Crear estudio con storage
study = optuna.create_study(
    study_name='rf_optimization',
    storage='sqlite:///optuna_study.db',  # Base de datos local
    load_if_exists=True  # Continuar si ya existe
)

# Entrenar
study.optimize(objective, n_trials=50)

# Cargar más tarde
study_loaded = optuna.load_study(
    study_name='rf_optimization',
    storage='sqlite:///optuna_study.db'
)
```

✅ **Ventaja**: Nunca pierdes progreso, incluso si se interrumpe.

---

In [0]:
print("\n" + "="*80)
print("EJEMPLO: PRUNING AUTOMÁTICO")
print("="*80)

from optuna.pruners import MedianPruner

# Función objetivo con pruning
def objective_with_pruning(trial):
    """
    Función objetivo que reporta valores intermedios para pruning.
    """
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'random_state': 42
    }
    
    # Entrenar incrementalmente y reportar valores intermedios
    from sklearn.model_selection import cross_validate
    
    model = RandomForestRegressor(**params)
    
    # CV con 3 folds, reportando resultados parciales
    scores = cross_val_score(
        model, X_train, y_train, 
        cv=3, 
        scoring='neg_mean_absolute_error'
    )
    
    # Reportar valores intermedios para pruning
    for step, score in enumerate(scores):
        trial.report(-score, step)
        
        # Verificar si debe hacer pruning
        if trial.should_prune():
            raise optuna.TrialPruned()
    
    return -scores.mean()

# Crear estudio con pruning
study_pruning = optuna.create_study(
    direction='minimize',
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=1)
)

print("\n🚀 Optimizando con pruning automático...")
print("   Trials que se vean poco prometedores serán detenidos temprano\n")

study_pruning.optimize(objective_with_pruning, n_trials=30, show_progress_bar=True)

print(f"\n📊 Resultados con Pruning:")
print(f"   Trials completos: {len([t for t in study_pruning.trials if t.state == optuna.trial.TrialState.COMPLETE])}")
print(f"   Trials podados: {len([t for t in study_pruning.trials if t.state == optuna.trial.TrialState.PRUNED])}")
print(f"   Mejor MAE: ${study_pruning.best_value:,.2f}")
print(f"\n   ⚙️ Ahorro: {len([t for t in study_pruning.trials if t.state == optuna.trial.TrialState.PRUNED])} trials detenidos temprano")

## 6️⃣ Ejercicios Prácticos

### 📝 Ejercicio 1: Optimizar Gradient Boosting

**Objetivo**: Usar Optuna para optimizar un modelo GradientBoostingRegressor.

**Tareas**:
1. Definir función objetivo para GradientBoostingRegressor
2. Hiperparámetros a optimizar:
   - `n_estimators`: 50-500
   - `learning_rate`: 0.01-0.3 (log scale)
   - `max_depth`: 3-10
   - `subsample`: 0.5-1.0
   - `min_samples_split`: 2-20
3. Ejecutar 50 trials
4. Comparar con Random Forest optimizado

**Pista**: Usa `trial.suggest_float('learning_rate', 0.01, 0.3, log=True)` para escala logarítmica.

---

### 📝 Ejercicio 2: Multi-objetivo

**Objetivo**: Optimizar **dos métricas simultáneamente**: MAE y tiempo de entrenamiento.

**Tareas**:
1. Modificar función objetivo para retornar tupla `(mae, tiempo)`
2. Crear estudio multi-objetivo: `optuna.create_study(directions=['minimize', 'minimize'])`
3. Analizar Pareto front (trade-off entre precisión y velocidad)
4. Seleccionar modelo según prioridad del negocio

---

### 📝 Ejercicio 3: Persistencia y Continuación

**Objetivo**: Guardar estudio en base de datos y continuar después.

**Tareas**:
1. Crear estudio con storage SQLite
2. Ejecutar 25 trials
3. "Interrumpir" (simular)
4. Cargar estudio y ejecutar 25 trials adicionales
5. Verificar que el total sea 50 trials

**Pista**: Usa `storage='sqlite:///mi_estudio.db'`

---

### 📝 Ejercicio 4: Features Espaciales

**Objetivo**: Incorporar features geoespaciales H3 y optimizar.

**Tareas**:
1. Agregar features espaciales al dataset:
   - `dist_sucursal_min`
   - `densidad_zona`
   - `facturacion_promedio_zona`
2. Re-optimizar Random Forest con features ampliadas
3. Comparar mejora vs. modelo sin features espaciales
4. Analizar importancia de features espaciales

---

## ✅ Conclusiones y Mejores Prácticas

### 🎯 Resumen del Módulo

**Lo que aprendimos**:

1. ✅ Diferencia entre **parámetros** e **hiperparámetros**
2. ✅ Métodos tradicionales: **Grid Search** y **Random Search**
3. ✅ **Optuna**: Búsqueda bayesiana eficiente
4. ✅ **Pruning**: Detener trials poco prometedores
5. ✅ **Visualizaciones**: Entender proceso de optimización
6. ✅ **Persistencia**: Guardar progreso en base de datos

---

### 💡 Mejores Prácticas

#### 1. **Definir Budget de Tiempo**

```python
# Opción A: Número de trials
study.optimize(objective, n_trials=100)

# Opción B: Tiempo límite
import datetime
timeout = 60 * 30  # 30 minutos
study.optimize(objective, timeout=timeout)
```

#### 2. **Usar Validación Cruzada**

✅ **BIEN**: Validación cruzada (más robusto)
```python
scores = cross_val_score(model, X_train, y_train, cv=5)
return -scores.mean()
```

❌ **MAL**: Train/validation split simple (puede overfittear)
```python
model.fit(X_train, y_train)
score = model.score(X_val, y_val)
return -score
```

#### 3. **Rangos Sensatos de Hiperparámetros**

✅ **BIEN**: Rangos informados
```python
'n_estimators': trial.suggest_int('n_estimators', 50, 300)  # Rango razonable
'max_depth': trial.suggest_int('max_depth', 3, 20)         # No demasiado profundo
```

❌ **MAL**: Rangos demasiado amplios
```python
'n_estimators': trial.suggest_int('n_estimators', 1, 10000)  # Muy amplio
'max_depth': trial.suggest_int('max_depth', 1, 1000)        # Excesivo
```

#### 4. **Escala Logarítmica para Learning Rate**

```python
# Para hiperparámetros que actúan en escala log
'learning_rate': trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True)
```

#### 5. **Guardar Mejor Modelo**

```python
# Al final de la optimización
import joblib

model_final = RandomForestRegressor(**study.best_params)
model_final.fit(X_train, y_train)

joblib.dump(model_final, 'best_model.pkl')
print("✅ Mejor modelo guardado")
```

---

### 🚀 Cuándo Usar Optuna

✅ **Usar Optuna cuando**:
- Tienes muchos hiperparámetros (>5)
- Grid Search es demasiado lento
- Necesitas optimización rápida
- Quieres visualizaciones automáticas
- Necesitas persistencia

❌ **NO usar Optuna cuando**:
- Solo 1-2 hiperparámetros (manual es más rápido)
- Dataset muy pequeño (<1000 registros)
- Modelo entrena en <1 segundo

---

### 📚 Recursos Adicionales

- **Documentación Oficial**: https://optuna.readthedocs.io/
- **Tutoriales**: https://optuna.org/#code_examples
- **Paper Original**: "Optuna: A Next-generation Hyperparameter Optimization Framework"
- **Comparaciones**: https://github.com/optuna/optuna-examples

---

## 🎓 ¡Felicitaciones!

**Has completado el módulo de Optimización de Hiperparámetros con Optuna.**

Ahora puedes:
- ✅ Optimizar modelos de forma eficiente
- ✅ Usar técnicas avanzadas (pruning, paralelización)
- ✅ Interpretar visualizaciones de optimización
- ✅ Aplicar mejores prácticas en proyectos reales

---

**Universidad del Aconcagua**  
**Laboratorio (Herramientas)**  
**Mendoza, Argentina**

# 🎯 Optimización de Hiperparámetros con Optuna
## Material Complementario - Laboratorio (Herramientas)
### Universidad del Aconcagua - Mendoza, Argentina

---

### 🎯 Objetivos de Aprendizaje

1. Comprender qué son los **hiperparámetros** y por qué optimizarlos
2. Conocer métodos tradicionales (**Grid Search**, **Random Search**)
3. Aprender a usar **Optuna** para optimización eficiente
4. Aplicar técnicas avanzadas: **pruning**, **paralelización**, **visualización**
5. Comparar resultados entre métodos

### 📁 Contenido

1. Introducción a Hiperparámetros
2. Métodos Tradicionales de Optimización
3. Introducción a Optuna
4. Ejemplo Práctico: Predicción de Ventas de Panadería
5. Técnicas Avanzadas
6. Ejercicios Prácticos

### ⏱️ Duración Estimada: 2 horas

---

## 1️⃣ ¿Qué son los Hiperparámetros?

### Diferencia: Parámetros vs. Hiperparámetros

#### 📊 **Parámetros**
- Son **aprendidos por el modelo** durante el entrenamiento
- Ejemplos: Pesos en redes neuronales, coeficientes en regresión lineal
- Se ajustan automáticamente con los datos

```python
# Ejemplo: En regresión lineal y = mx + b
m, b  # <- PARÁMETROS (aprendidos)
```

#### ⚙️ **Hiperparámetros**
- Son **configuraciones externas** que controlamos ANTES del entrenamiento
- Ejemplos: Número de árboles en Random Forest, learning rate, profundidad máxima
- **NO** se aprenden automáticamente, los definimos nosotros

```python
# Ejemplo: Random Forest
RandomForestClassifier(
    n_estimators=100,      # <- HIPERPARÁMETRO
    max_depth=10,          # <- HIPERPARÁMETRO
    min_samples_split=5    # <- HIPERPARÁMETRO
)
```

---

### 🎯 Por qué Importan los Hiperparámetros

Los hiperparámetros tienen **impacto directo** en:

1. **Precisión del modelo**: Configuración incorrecta → bajo rendimiento
2. **Overfitting/Underfitting**: 
   - `max_depth` muy alto → overfitting
   - `max_depth` muy bajo → underfitting
3. **Tiempo de entrenamiento**: Más árboles = más tiempo
4. **Memoria utilizada**: Modelos más complejos = más RAM

**Ejemplo real**:

| max_depth | Accuracy Train | Accuracy Test | Conclusión |
|-----------|----------------|---------------|------------|
| 3 | 75% | 74% | 🟡 Underfitting |
| 10 | 92% | 89% | ✅ Equilibrado |
| 50 | 100% | 78% | 🔴 Overfitting |

✅ **Conclusión**: Encontrar los hiperparámetros óptimos es crucial.

---

## 2️⃣ Métodos Tradicionales de Optimización

### 🔲 Grid Search (Búsqueda Exhaustiva)

**Concepto**: Prueba **todas las combinaciones posibles** de hiperparámetros.

```python
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, 20],
    'min_samples_split': [2, 5, 10]
}

# Total de combinaciones: 3 x 4 x 3 = 36 modelos a entrenar
```

✅ **Ventajas**:
- Garantiza encontrar la mejor combinación en el grid
- Fácil de entender e implementar

❌ **Desventajas**:
- 🐢 **Extremadamente lento** con muchos hiperparámetros
- Explosión combinatoria: 10 parámetros con 5 valores = 9,765,625 combinaciones!
- Desperdicia recursos en zonas poco prometedoras

---

### 🎲 Random Search (Búsqueda Aleatoria)

**Concepto**: Prueba combinaciones **aleatorias** de hiperparámetros.

```python
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [50, 100, 150, 200, 250],
    'max_depth': [5, 10, 15, 20, 25, 30],
    'min_samples_split': [2, 5, 10, 15]
}

# Prueba solo 20 combinaciones aleatorias (de miles posibles)
random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_dist,
    n_iter=20  # Número de combinaciones a probar
)
```

✅ **Ventajas**:
- Más rápido que Grid Search
- Explora más espacio con menos iteraciones

❌ **Desventajas**:
- No aprende de iteraciones anteriores
- Puede perder combinaciones óptimas
- Aun puede ser lento

---

### 📈 Comparación Visual

**Grid Search vs. Random Search**:

```
Grid Search (exhaustivo):        Random Search:

  [X] [X] [X] [X] [X]              [ ] [X] [ ] [ ] [X]
  [X] [X] [X] [X] [X]              [X] [ ] [ ] [X] [ ]
  [X] [X] [X] [X] [X]              [ ] [ ] [X] [ ] [ ]
  [X] [X] [X] [X] [X]              [ ] [X] [ ] [X] [ ]
  [X] [X] [X] [X] [X]              [X] [ ] [ ] [ ] [X]
```

✅ Random Search cubre más área con menos iteraciones

---

## 3️⃣ Introducción a Optuna

### 🧠 ¿Qué es Optuna?

**Optuna** es un framework de optimización de hiperparámetros que usa **búsqueda bayesiana** y **pruning automático**.

#### Características Clave

1. ⚙️ **Optimización Automática**: Decide qué combinaciones probar próximamente
2. ✏️ **Pruning Inteligente**: Detiene entrenamientos poco prometedores
3. 🚀 **Paralelización**: Ejecuta múltiples trials simultáneamente
4. 📊 **Visualización**: Gráficos de optimización y relaciones entre hiperparámetros
5. 💾 **Persistencia**: Guarda progreso en base de datos

---

### 🔍 Cómo Funciona Optuna

#### Búsqueda Bayesiana Simplificada

Optuna usa **Tree-structured Parzen Estimator (TPE)** por defecto:

1. 🎯 **Inicio**: Prueba combinaciones aleatorias
2. 🧠 **Aprende**: Modela qué hiperparámetros funcionan mejor
3. 🎯 **Explota**: Sugiere valores en zonas prometedoras
4. 🔄 **Explora**: Ocasionalmente prueba zonas nuevas
5. 🔁 **Repite**: Converge hacia óptimo

**Ventaja**: Aprende de cada trial y sugiere mejores valores.

---

### 🎯 Ventajas de Optuna vs. Métodos Tradicionales

| Característica | Grid Search | Random Search | Optuna |
|------------------|-------------|---------------|--------|
| Velocidad | 🐢 Muy lento | 🐇 Rápido | 🚀 Muy rápido |
| Aprende de trials | ❌ No | ❌ No | ✅ Sí |
| Pruning automático | ❌ No | ❌ No | ✅ Sí |
| Paralelización | ✅ Sí | ✅ Sí | ✅ Sí |
| Fácil uso | ✅ Sí | ✅ Sí | ✅ Sí |
| Visualizaciones | ❌ No | ❌ No | ✅ Sí |

---

### 📦 Instalación

```python
%pip install optuna --quiet
```

---

In [0]:
# Instalar Optuna
%pip install optuna==3.5.0 --quiet

# Importar librerías
import pandas as pd
import numpy as np
import optuna
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ Librerías importadas")
print(f"   Optuna versión: {optuna.__version__}")

In [0]:
# Cargar datasets
ruta_datos = '/Workspace/Users/cortega@uda.edu.ar/Laboratorio/Datasets/'

df_ventas = pd.read_csv(ruta_datos + 'ventas.csv')
df_clientes = pd.read_csv(ruta_datos + 'clientes.csv')
df_productos = pd.read_csv(ruta_datos + 'productos.csv')
df_detalles = pd.read_csv(ruta_datos + 'detalles_ventas.csv')

print("✅ Datasets cargados")
print(f"   Ventas: {len(df_ventas):,}")
print(f"   Clientes: {len(df_clientes):,}")
print(f"   Productos: {len(df_productos):,}")

In [0]:
print("="*80)
print("PREPARACIÓN DE DATOS PARA MODELO")
print("="*80)

# Convertir fecha
df_ventas['fecha'] = pd.to_datetime(df_ventas['fecha'])

# Crear features temporales
df_ventas['dia_semana'] = df_ventas['fecha'].dt.dayofweek
df_ventas['dia_mes'] = df_ventas['fecha'].dt.day
df_ventas['mes'] = df_ventas['fecha'].dt.month
df_ventas['es_fin_de_semana'] = df_ventas['dia_semana'].isin([5, 6]).astype(int)

# Unir con clientes para features
df_ml = df_ventas[df_ventas['cliente_id'].notna()].merge(
    df_clientes[['cliente_id', 'segmento']], 
    on='cliente_id',
    how='left'
)

# Codificar categorías
df_ml['segmento_encoded'] = df_ml['segmento'].astype('category').cat.codes

# Features finales
features = [
    'sucursal_id', 
    'dia_semana', 
    'dia_mes', 
    'mes', 
    'es_fin_de_semana',
    'segmento_encoded'
]

target = 'total'

X = df_ml[features]
y = df_ml[target]

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\n📊 Datos preparados:")
print(f"   Train: {len(X_train):,} registros")
print(f"   Test:  {len(X_test):,} registros")
print(f"   Features: {len(features)}")
print(f"   Target: {target} (predicción de monto de venta)")

In [0]:
print("\n" + "="*80)
print("BASELINE: MODELO SIN OPTIMIZAR")
print("="*80)

# Entrenar modelo con hiperparámetros por defecto
model_baseline = RandomForestRegressor(random_state=42)
model_baseline.fit(X_train, y_train)

# Evaluar
y_pred_baseline = model_baseline.predict(X_test)

mae_baseline = mean_absolute_error(y_test, y_pred_baseline)
rmse_baseline = np.sqrt(mean_squared_error(y_test, y_pred_baseline))
r2_baseline = r2_score(y_test, y_pred_baseline)

print(f"\n📊 Métricas Baseline (hiperparámetros por defecto):")
print(f"   MAE:  ${mae_baseline:,.2f}")
print(f"   RMSE: ${rmse_baseline:,.2f}")
print(f"   R²:   {r2_baseline:.4f}")

print(f"\n⚙️ Hiperparámetros usados:")
print(f"   n_estimators: {model_baseline.n_estimators}")
print(f"   max_depth: {model_baseline.max_depth}")
print(f"   min_samples_split: {model_baseline.min_samples_split}")
print(f"   min_samples_leaf: {model_baseline.min_samples_leaf}")

### 🎯 Definir Función Objetivo para Optuna

La función objetivo:
1. Recibe un `trial` (sugiere hiperparámetros)
2. Entrena el modelo
3. Evalúa con validación cruzada
4. Retorna la métrica a **minimizar** (MAE en este caso)

**Importante**: Optuna **minimiza** por defecto. Para maximizar (ej. R²), retornar `-r2`.

In [0]:
def objective(trial):
    """
    Función objetivo para Optuna.
    Sugiere hiperparámetros, entrena modelo y retorna MAE.
    """
    
    # Sugerir hiperparámetros
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'random_state': 42
    }
    
    # Crear modelo con hiperparámetros sugeridos
    model = RandomForestRegressor(**params)
    
    # Validación cruzada (3-fold para velocidad)
    scores = cross_val_score(
        model, 
        X_train, 
        y_train, 
        cv=3, 
        scoring='neg_mean_absolute_error',
        n_jobs=-1
    )
    
    # Retornar MAE promedio (valor absoluto)
    mae = -scores.mean()
    
    return mae

print("✅ Función objetivo definida")

In [0]:
print("\n" + "="*80)
print("OPTIMIZACIÓN CON OPTUNA")
print("="*80)

# Crear estudio
study = optuna.create_study(
    direction='minimize',  # Minimizar MAE
    study_name='rf_optimization',
    sampler=optuna.samplers.TPESampler(seed=42)  # Tree-structured Parzen Estimator
)

print(f"\n🚀 Iniciando optimización...")
print(f"   Algoritmo: TPE (Tree-structured Parzen Estimator)")
print(f"   Métrica: MAE (minimizar)")
print(f"   Trials: 50")
print(f"\n   Progreso:")

# Ejecutar optimización
study.optimize(
    objective, 
    n_trials=50,  # Número de combinaciones a probar
    show_progress_bar=True
)

print(f"\n✅ Optimización completada")